<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####1. Requirement
Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:

#offline_students_schema = "ID string, FirstName string, LastName string, Address string, Skills string, Contacts string"
offline_students_schema = "ID string, first_name string, last_name string, Address string, Skills string, Contacts string, FirstName string, LastName string"

offline_students_raw_df1 = (
    spark.read.format("csv")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .schema(offline_students_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/students_offline.csv")
)

#offline_students_raw_df.display()
offline_students_raw_df1.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")
#offline_students_raw_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_var_students")

##### Fixes I must do since the cells were not running according to the video. it seems a file ws not updated before uploading it to the training material so I needed to remove two columns adn modify two more

In [0]:
offline_students_raw_df= (offline_students_raw_df1.drop("FirstName")
                                                   .drop("LastName")
                                                   .withColumnRenamed("first_name", "FirstName")
                                                   .withColumnRenamed("last_name", "LastName")
                           )
offline_students_raw_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-8462731191287876>, line 1
----> 1 offline_students_raw_df= (offline_students_raw_df1.drop("FirstName")
      2                                                    .drop("LastName")
      3                                                    .withColumnRenamed("first_name", "FirstName")
      4                                                    .withColumnRenamed("last_name", "LastName")
      5                            )
      6 offline_students_raw_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students_raw")

NameError: name 'offline_students_raw_df1' is not defined

####2. Analysis Requirement
We want to know country wise student count.

In [0]:
%sql

with offline_students(
                      select id, from_json(address,
                                            """struct< AddressLine1 string,
                                                       AddressLine2 string,
                                                       City string,
                                                       Country string,
                                                       Pin string,
                                                       State string
                                                     >
                      """) as address
  from dev.spark_db.offline_students_raw
)
select address.country, count(*) as count
from offline_students
group by address.country

country,count
India,3
Engaland,1
Northern Ireland,1


####3. Requirement
Prepare an offline_students table which is ready for analysis

Complex Data Types in Spark
1. Struct
2. Array
3. Map

In [0]:
from pyspark.sql.functions import from_json

address_schema = "struct<AddressLine1 string, AddressLine2 string, City string, Country string, Pin string, State string>"
skills_schema = "array<struct<Skill string, YearsOfExperience string>>"
contacts_schema = "map<string, string>"

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": from_json("address", address_schema),
        "skills": from_json("skills", skills_schema),
        "contacts": from_json("contacts", contacts_schema)
    })
)

#offline_students_df.display()
offline_students_df.write.mode("overwrite").saveAsTable("dev.spark_db.offline_students")

####4. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

4.1 What is country wise student count.

In [0]:
%sql

select address.Country, count(*) as count
from dev.spark_db.offline_students
group by address.Country

Country,count
India,3
Engaland,1
Northern Ireland,1


4.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql

with offline_students_skills(
   select id, FirstName, LastName, explode(skills) as skills
   from dev.spark_db.offline_students
)
select id, firstname, lastname, skills.*
from offline_students_skills
where skills.Skill like "%Spark%" and skills.YearsOfExperience > 1

id,firstname,lastname,Skill,YearsOfExperience
101,Prashant,Pandey,Apache Spark,5
104,Nasima,Khatun,Apache Spark,2
105,Pritam,Jain,Apache Spark,3


4.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql

select ID, FirstName, LastName, contacts['email']
from dev.spark_db.offline_students
where contacts['phone'] is null and contacts['whatsapp'] is null

ID,FirstName,LastName,contacts[email]
103,Katie,Mcloskey,ert89@abc.com
104,Nasima,Khatun,magt23@abc.com


&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>